In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.6 Scaling laws

Given a fixed compute budget, how big should the model be and how much data should
it see? Scaling laws (Chinchilla) say loss falls predictably with parameters and
tokens, and that the two should grow together. Here we just watch a bigger model
reach a lower loss on the same data.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

def train_size(n_embd, n_layer, steps):
    torch.manual_seed(0)
    cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32,
                         n_layer=n_layer, n_head=2, n_embd=n_embd)
    model = init_weights(PocketLM(cfg))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(0)
    for _ in range(steps):
        x, y = make_batch(data, 32, 16, generator=g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    params = sum(p.numel() for p in model.parameters())
    return params, estimate_loss(model, data, 32, 16, iters=10, generator=g)

steps = 120 if os.environ.get("POCKETLM_CI") else 400
small = train_size(32, 1, steps)
big = train_size(96, 3, steps)
print(f"small: {small[0]:>6} params, loss {small[1]:.3f}")
print(f"big:   {big[0]:>6} params, loss {big[1]:.3f}")

More capacity, lower loss, at the cost of more compute. The engineering job is
spending a fixed budget where it buys the most: scaling laws turn that into math.

In [ ]:
assert big[0] > small[0]                 # the big model has more parameters
import math
assert small[1] < math.log(tok.vocab_size) and big[1] < math.log(tok.vocab_size)